In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

import joblib

In [ ]:
df_treino = joblib.load('')
df_treino

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Datasets/deteccao-de-pos/conjuntoDeDados_treinamento.joblib'

In [ ]:
df_treino['target_id'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 2008 entries, 0 to 2007
Series name: target_id
Non-Null Count  Dtype
--------------  -----
2008 non-null   int64
dtypes: int64(1)
memory usage: 15.8 KB


In [ ]:
df_treino['target_id'].unique()

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143,
       144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156,
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169,
       170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 18

In [ ]:
# Features: a embedding do comentário, a do alvo e a parent_label.

emb_atual = np.array(df_treino['embedding'].tolist())

# Embedding do Alvo
# vou separar em dataframes por target_id (cada thread). Deixará depois os diferentes jeitos de separação de features mais facil
# Cria um dicionário: chave = target_id, valor = DataFrame da thread
threads = {target_id: group for target_id, group in df_treino.groupby('target_id')}
threads.keys() # deu certo

dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194])

In [ ]:
# mapear msg alvo (id='Alvo' definido no pre-process) pra cada thread (target_id)
target_id_to_emb_alvo = {}
for target_id, thread_df in threads.items():
    target_id_to_emb_alvo[target_id] = thread_df[thread_df['id'] == 'Alvo']['embedding'].values[0]

# lista das emb_alvo alinhada com as linhas do df confomre
emb_alvos = []
for target_id in df_treino['target_id']:
    emb_alvos.append(target_id_to_emb_alvo[target_id])
emb_alvos = np.array(emb_alvos)

""" 
# método mais direto:
# mapping para obtermos embeddings do alvo (id='Alvo') e associarmos para cada linha conforme sua thread (target_id)
target_id_to_emb_alvo = {target_id: thread_df[thread_df['id'] == 'Alvo']['embedding'].values[0] for target_id, thread_df in threads.items()}

# array das embeddings alinhadas
emb_alvos = np.array([target_id_to_emb_alvo[target_id] for target_id in df_treino['target_id']])
 """

print(len(emb_alvos))

194


In [ ]:
# CONCATENAR AS FEATURES
# precisamos que tenham o mesmo número de linhas (e dimnesões compatíveis)
parent_label = df_treino['parent_label_enc'].values.reshape(-1, 1) # Alterna estrutura pra 2D



# FEATURES
X_combined = np.concatenate((emb_alvos, emb_atual, parent_label), axis=1)

# Verificando as dimensões
print("Shape of emb_alvos:", emb_alvos.shape)
print("Shape of parent_label:", parent_label.shape)
print("Shape of emb_atual:", emb_atual.shape)
print("Shape of X_combined:", X_combined.shape)

# vou trazer esses pro pre

Shape of emb_alvos_aligned: (2008, 768)
Shape of parent_label: (2008, 1)
Shape of emb_atual: (2008, 768)
Shape of X_combined: (2008, 1537)


In [ ]:
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Remove rows where 'label_enc' is NaN
df_treino_cleaned = df_treino.dropna(subset=['label_enc']).copy()

# Align the features X_combined with the cleaned dataframe
X_combined_cleaned = X_combined[df_treino['label_enc'].notna()]

modelo_rf.fit(X_combined_cleaned, df_treino_cleaned['label_enc'])

RandomForestClassifier(random_state=42)

In [ ]:
# importar teste
df_teste = joblib.load('/content/drive/MyDrive/Datasets/deteccao-de-pos/conjuntoDeDados_teste.joblib')
df_teste

,target_id,target_message,id,parent_id,author,parent_name,parent_message,message,parent_label,target_parent_message,target_and_message,label,embedding,label_enc,parent_label_enc
0,195,Eu acho que a questão do aborto é ainda mais c...,l34sq0z,1cn3qlu,Matthew_Davenport,Raiz,NaN,Mulher gravida tem preferencia e adoração em t...,Discorda,Eu acho que a questão do aborto é ainda mais c...,Eu acho que a questão do aborto é ainda mais c...,Concorda,"[-0.20494509, 0.17932542, 0.47428623, -0.21237...",1.0,-1.0
1,195,Eu acho que a questão do aborto é ainda mais c...,l34ubkr,l34sq0z,Danielle_Cannon,Matthew_Davenport,Mulher gravida tem preferencia e adoração em t...,"A questão não é sobre se o feto é, ou não, viv...",Concorda,Eu acho que a questão do aborto é ainda mais c...,Eu acho que a questão do aborto é ainda mais c...,Discorda,"[0.1266863, 0.32740223, 0.515734, 0.1914688, 0...",-1.0,1.0
2,195,Eu acho que a questão do aborto é ainda mais c...,l34vauq,l34ubkr,Matthew_Davenport,Danielle_Cannon,"A questão não é sobre se o feto é, ou não, viv...","Não, é um ser vivo superior ja que vai ser um...",Discorda,Eu acho que a questão do aborto é ainda mais c...,Eu acho que a questão do aborto é ainda mais c...,Concorda,"[-0.18790433, -0.023872852, 0.666812, 0.399194...",1.0,-1.0
3,195,Eu acho que a questão do aborto é ainda mais c...,l34vvxa,l34vauq,Danielle_Cannon,Matthew_Davenport,"Não, é um ser vivo superior ja que vai ser um...","Certo, ele \""vai ser\"" um humano, portanto não...",Concorda,Eu acho que a questão do aborto é ainda mais c...,Eu acho que a questão do aborto é ainda mais c...,Discorda,"[0.26420322, -0.036678404, 0.63467455, -0.1257...",-1.0,1.0
4,195,Eu acho que a questão do aborto é ainda mais c...,l34wymg,l34vvxa,Matthew_Davenport,Danielle_Cannon,"Certo, ele \""vai ser\"" um humano, portanto não...","Ele ja ta em curso, camarada, voce vai interro...",Discorda,Eu acho que a questão do aborto é ainda mais c...,Eu acho que a questão do aborto é ainda mais c...,Concorda,"[-0.018243317, 0.020791816, 0.74743634, 0.2494...",1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
502,245,Eu entendo que a gente está lidando com uma si...,Alvo,NaN,NaN,NaN,NaN,Eu entendo que a gente está lidando com uma si...,NaN,NaN,NaN,NaN,"[0.18799467, -0.23550084, 0.55626553, -0.06233...",NaN,NaN
503,246,Sobre essa discussão das regras do pix e o que...,Alvo,NaN,NaN,NaN,NaN,Sobre essa discussão das regras do pix e o que...,NaN,NaN,NaN,NaN,"[-0.12828174, -0.068098806, 0.51325995, -0.173...",NaN,NaN
504,247,"Essa discussão sobre as novas regras do pix, e...",Alvo,NaN,NaN,NaN,NaN,"Essa discussão sobre as novas regras do pix, e...",NaN,NaN,NaN,NaN,"[-0.2339915, 0.018618952, 0.29558563, 0.001226...",NaN,NaN
505,248,"Isso q tentei explicar pra meu cunhado minion,...",Alvo,NaN,NaN,NaN,NaN,"Isso q tentei explicar pra meu cunhado minion,...",NaN,NaN,NaN,NaN,"[-0.041741066, -0.3309277, 0.13098104, -0.2209...",NaN,NaN


In [ ]:
# Features for the test set: embedding do comentário, a do alvo e a parent_label.

emb_atual_teste = np.array(df_teste['embedding'].tolist())

# Embedding do Alvo para o test set
# Need to separate into dataframes by target_id for the test set
threads_teste = {target_id: group for target_id, group in df_teste.groupby('target_id')}

# Create emb_alvos_teste with the same number of rows as df_teste
# Map each target_id to its 'Alvo' embedding
target_id_to_emb_alvo_teste = {target_id: thread_df[thread_df['id'] == 'Alvo']['embedding'].values[0] for target_id, thread_df in threads_teste.items()}

# Create a new 'emb_alvos_teste' array by mapping the 'target_id' of each row in df_teste to its corresponding 'Alvo' embedding
emb_alvos_aligned_teste = np.array([target_id_to_emb_alvo_teste[target_id] for target_id in df_teste['target_id']])


# Extract 'parent_label_enc' from the test set and convert to numpy array, reshaping to have a second dimension
parent_label_teste = df_teste['parent_label_enc'].values.reshape(-1, 1)

# Concatenate the features for the test set
X_test_combined = np.concatenate((emb_alvos_aligned_teste, parent_label_teste, emb_atual_teste), axis=1)

print("Shape of emb_alvos_aligned_teste:", emb_alvos_aligned_teste.shape)
print("Shape of parent_label_teste:", parent_label_teste.shape)
print("Shape of emb_atual_teste:", emb_atual_teste.shape)
print("Shape of X_test_combined:", X_test_combined.shape)

Shape of emb_alvos_aligned_teste: (507, 768)
Shape of parent_label_teste: (507, 1)
Shape of emb_atual_teste: (507, 768)
Shape of X_test_combined: (507, 1537)


In [ ]:
from sklearn.metrics import classification_report

# Remove rows from the test set where 'label_enc' is NaN to get the true labels
df_teste_cleaned = df_teste.dropna(subset=['label_enc']).copy()

# Align the test features with the cleaned test dataframe
X_test_combined_cleaned = X_test_combined[df_teste['label_enc'].notna()]
y_test_true = df_teste_cleaned['label_enc']

# Make predictions on the cleaned test features
y_test_pred = modelo_rf.predict(X_test_combined_cleaned)

# Print the classification report
print(classification_report(y_test_true, y_test_pred))

              precision    recall  f1-score   support

        -1.0       0.40      0.18      0.25       101
         0.0       0.84      0.26      0.40       190
         1.0       0.40      0.86      0.54       161

    accuracy                           0.45       452
   macro avg       0.55      0.43      0.39       452
weighted avg       0.59      0.45      0.41       452

